---
**Script name:** 04_03_06_manual_edits.ipynb

**Purpose of script:** This script is to manually edit a few stand alone Nodes which could not be corrected using the KL Divergence methodology.

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths. Change 'postgresql://user:@localhost/db_name' to PostgreSQL databse login credentials.

---

## Importing libraries that are required

In [1]:
import time
from datetime import timedelta
from scipy.special import rel_entr
from scipy.stats import entropy
import sqlalchemy as psql
import re
import pickle
import pandas as pd
import numpy as np
import os

## Define variables

# set path to raw node data
dir_path = Path('path_to_folder')
input_dir = Path("../../04_01_data_wrangling/output")
output_dir = Path(".")

## Functions that are used throughout the script

In [7]:
def edit_dataframe(df, change_type, node_name):
    new_df = df.assign(method = change_type)
    new_df = new_df.assign(data_size = len(df))
    new_df = new_df.assign(path = 'python')
    new_df = new_df.assign(radio_id = 5)
    new_df = new_df.assign(node_id = node_name)
    new_df = new_df.assign(validated = 0)
    new_df = new_df.assign(station_id = None)
    
    return new_df

In [9]:
def edit_dataframe(df, change_type, node_name):
    new_df = df.assign(method = change_type)
    new_df = new_df.assign(data_size = None)
    new_df = new_df.assign(path = 'C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_Motus/UK_Bats_Roost_Node_Data/3750b3 - Rock Farm/beep_1.csv')
    new_df = new_df.assign(radio_id = 5)
    new_df = new_df.assign(node_id = node_name)
    new_df = new_df.assign(validated = 0)
    new_df = new_df.assign(station_id = None)
    
    return new_df

## Load Data

In [ ]:
#read in dataframes that have mistakes
brookwood = pd.read_csv(f'{dir_path}/32b6e0 - Brookwood Mines/beep_4.csv')
rock_farm = pd.read_csv(f'{dir_path}/3750b3 - Rock Farm/beep_1.csv')
lower_upcott = pd.read_csv(f'{dir_path}/327d6b - Lower Upcott/beep_1.csv')

## Data processing

In [3]:
#change time column to datetime format
brookwood['time'] = pd.to_datetime(brookwood['time'], format = '%Y-%m-%dT%H:%M:%SZ')
rock_farm['time'] = pd.to_datetime(rock_farm['time'], format = '%Y-%m-%dT%H:%M:%SZ')
lower_upcott['time'] = pd.to_datetime(lower_upcott['time'], format = '%Y-%m-%dT%H:%M:%SZ')

In [4]:
#split rock farm node into mislabelled and correctly labelled data
rock_farm_mislabelled = rock_farm.loc[(rock_farm['time'] <= '2018-01-01 00:00:00')]
rock_farm_correct = rock_farm.loc[(rock_farm['time'] >= '2018-01-01 00:00:00')]

In [5]:
#set correction values
brookwood_correction = pd.Timedelta(hours=53676, minutes=37, seconds=5)
rock_farm_correction = pd.Timedelta(hours=53698, minutes=1, seconds=7)
lower_upcott_correction = pd.Timedelta(hours=59893, minutes=25, seconds=50)

In [6]:
#add the correction value manually
brookwood['time'] = brookwood['time'] + brookwood_correction
rock_farm_mislabelled['time'] = rock_farm_mislabelled['time'] + rock_farm_correction
lower_upcott['time'] = lower_upcott['time'] + lower_upcott_correction

C:\Users\katie\AppData\Local\Temp\ipykernel_25856\4044616631.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rock_farm_mislabelled['time'] = rock_farm_mislabelled['time'] + rock_farm_correction


In [8]:
#edit the corrected dataframes
brookwood_new = edit_dataframe(brookwood, 'manual', '32b6e0')
rock_farm_new = edit_dataframe(rock_farm_mislabelled, 'manual', '3750b3')
lower_upcott_new = edit_dataframe(lower_upcott, 'manual', '327d6b')

In [10]:
#edit correct data
rock_farm_correct_new = edit_dataframe(rock_farm_correct, 'ctt', '3750b3')

In [11]:
#join all node data into one dataframe to input into sql 
manual_data = pd.concat([brookwood_new, rock_farm_new, rock_farm_correct_new, lower_upcott_new])
manual_data

,time,id,rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-02-15 13:38:11,55331961,-109,manual,873,python,5,32b6e0,0,None
1,2023-02-15 13:38:20,55331961,-102,manual,873,python,5,32b6e0,0,None
2,2023-02-15 13:38:29,55331961,-109,manual,873,python,5,32b6e0,0,None
3,2023-02-15 13:38:38,55331961,-108,manual,873,python,5,32b6e0,0,None
4,2023-02-15 13:38:48,55331961,-101,manual,873,python,5,32b6e0,0,None
...,...,...,...,...,...,...,...,...,...,...
1558,2024-01-12 22:08:26,4B7FAA34,-112,manual,1563,python,5,327d6b,0,None
1559,2024-01-19 18:46:30,80000000,-108,manual,1563,python,5,327d6b,0,None
1560,2024-01-19 19:41:53,34000000,-100,manual,1563,python,5,327d6b,0,None
1561,2024-01-19 20:25:49,34190080,-114,manual,1563,python,5,327d6b,0,None


In [12]:
#edit column names to match sql table
manual_data.rename(columns = {"id": "tag_id", "rssi": "tag_rssi"}, inplace=True)
manual_data

,time,tag_id,tag_rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-02-15 13:38:11,55331961,-109,manual,873,python,5,32b6e0,0,None
1,2023-02-15 13:38:20,55331961,-102,manual,873,python,5,32b6e0,0,None
2,2023-02-15 13:38:29,55331961,-109,manual,873,python,5,32b6e0,0,None
3,2023-02-15 13:38:38,55331961,-108,manual,873,python,5,32b6e0,0,None
4,2023-02-15 13:38:48,55331961,-101,manual,873,python,5,32b6e0,0,None
...,...,...,...,...,...,...,...,...,...,...
1558,2024-01-12 22:08:26,4B7FAA34,-112,manual,1563,python,5,327d6b,0,None
1559,2024-01-19 18:46:30,80000000,-108,manual,1563,python,5,327d6b,0,None
1560,2024-01-19 19:41:53,34000000,-100,manual,1563,python,5,327d6b,0,None
1561,2024-01-19 20:25:49,34190080,-114,manual,1563,python,5,327d6b,0,None


In [13]:
#remove index from dataframe
manual_data = manual_data.reset_index(drop=True)

In [14]:
#set possible first and last dates
brookwood_first_time = pd.to_datetime('2023-02-15 13:35:11',
                                    format='%Y-%m-%d %H:%M:%S')
brookwood_last_time = pd.to_datetime('2023-09-07 16:20:00',
                                    format='%Y-%m-%d %H:%M:%S')
rock_farm_first_time = pd.to_datetime('2023-02-16 09:49:28',
                                    format='%Y-%m-%d %H:%M:%S')
rock_farm_last_time = pd.to_datetime('2023-04-07 20:11:22',
                                    format='%Y-%m-%d %H:%M:%S')
lower_upcott_first_time = pd.to_datetime('2023-11-01 14:20:20',
                                    format='%Y-%m-%d %H:%M:%S')
lower_upcott_last_time = pd.to_datetime('2024-01-22 09:27:00',
                                    format='%Y-%m-%d %H:%M:%S')

In [15]:
#calculate the time difference
brookwood_first_time_delta = brookwood_new['time'].iloc[1] - brookwood_first_time
brookwood_last_time_delta = brookwood_last_time - brookwood_new['time'].iloc[-1]
rock_farm_first_time_delta = rock_farm_new['time'].iloc[1] - rock_farm_first_time
rock_farm_last_time_delta = rock_farm_last_time - rock_farm_new['time'].iloc[-1]
lower_upcott_first_time_delta = lower_upcott_new['time'].iloc[1] - lower_upcott_first_time
lower_upcott_last_time_delta = lower_upcott_last_time - lower_upcott_new['time'].iloc[-1]

In [16]:
#convert to seconds
brookwood_first_time_delta_seconds = brookwood_first_time_delta.total_seconds()
brookwood_last_time_delta_seconds = brookwood_last_time_delta.total_seconds()
rock_farm_first_time_delta_seconds = rock_farm_first_time_delta.total_seconds()
rock_farm_last_time_delta_seconds = rock_farm_last_time_delta.total_seconds()
lower_upcott_first_time_delta_seconds = lower_upcott_first_time_delta.total_seconds()
lower_upcott_last_time_delta_seconds = lower_upcott_last_time_delta.total_seconds()

In [17]:
#set conditions to join on depending on dataframe size
conditions = [
    (manual_data['node_id'] == '32b6e0'),
    (manual_data['node_id'] == '3750b3'),
    (manual_data['node_id'] == '327d6b')
]

results = [max(brookwood_first_time_delta_seconds, brookwood_last_time_delta_seconds), 
           max(rock_farm_first_time_delta_seconds, rock_farm_last_time_delta_seconds), 
           max(lower_upcott_first_time_delta_seconds, lower_upcott_last_time_delta_seconds)]

#create merging column
manual_data['max_error_seconds'] = np.select(conditions, results)

## Export Data

In [ ]:
#connect to postgres database
engine = psql.create_engine('postgresql://user:@localhost/db_name')

In [20]:
#send data to sql table
manual_data.to_sql('manual_edited_node_data',
                   con=engine,
                   if_exists='append',
                   index=False)

297